# Assignment 2, notebook 3b: Experiments that did not make the final model

Each experiment below asks "would X improve the model?" and is scored the same way: train FY2014-17, test on the real FY2019 loans (6.6% bad). Run notebook 3 first (this notebook reads its tuned settings).

1. Resampling the training data (and why a balanced test set is misleading)
2. Other definitions of a bad loan
3. Outside data: state unemployment rates (BLS)
4. Outside data: an older SBA snapshot (as of 31 Dec 2022) - proof of the term-field leak, borrower and lender history, more training years

**Data needed for section 4:** the three SBA FOIA files `foia-7afy1991-fy1999-asof-221231.csv`, `foia-7afy2000-fy2009-asof-221231.csv`, `foia-7afy2010-fy2019-asof-221231.csv` in `data/external/`.

In [11]:
import json, warnings, urllib.request
import numpy as np
import pandas as pd
import plotly.express as px
from pathlib import Path
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve
import lightgbm as lgb
warnings.filterwarnings("ignore")


In [12]:
TEAL, GREY, RED = "#0E7C86", "#B8BFCC", "#C0392B"
PROC = Path("../../data/processed"); OUT = Path("../outputs"); EXT = Path("../../data/external")
tr, va, te = (pd.read_parquet(PROC / f"a2_{s}.parquet") for s in ["train", "valid", "test"])
FEATS = sum(json.load(open(OUT / "a2_feature_groups.json")).values(), [])
cfg = json.load(open(OUT / "a2_final_model_config.json"))
PARAMS = dict(n_estimators=cfg["n_trees"], subsample_freq=1, random_state=42, n_jobs=-1, verbose=-1, **cfg["params"])
def model(): return lgb.LGBMClassifier(**PARAMS)

def score(y, p):
    pr, rc, _ = precision_recall_curve(y, p)
    return {"AUC": roc_auc_score(y, p), "PR-AUC": average_precision_score(y, p),
            "PR-AUC / random": average_precision_score(y, p) / y.mean(),
            "best point where precision = recall": float(np.minimum(pr, rc).max()),
            "precision when recall >= 70%": float(pr[rc >= 0.7].max())}

def fit_score(fs, Xtr=None, ytr=None, Xte=None, yte=None):
    Xtr = tr if Xtr is None else Xtr; Xte = te if Xte is None else Xte
    ytr = Xtr["bad"] if ytr is None else ytr; yte = Xte["bad"] if yte is None else yte
    return score(yte, model().fit(Xtr[fs], ytr).predict_proba(Xte[fs])[:, 1])

base = fit_score(FEATS)
pd.Series(base, name="baseline (all features)").round(3).to_frame()

,baseline (all features)
AUC,0.754
PR-AUC,0.198
PR-AUC / random,2.999
best point where precision = recall,0.242
precision when recall >= 70%,0.130


## 1. Resampling the training data
*Question:* does undersampling the good loans (so the model sees a higher share of bad ones) improve the model?
*Why it might not:* resampling changes how often each class is seen, not what the model can tell apart. It shifts the probabilities but usually not the ranking.

In [13]:
bad_tr, good_tr = tr[tr.bad == 1], tr[tr.bad == 0]
rows = [{"training data": f"No resampling ({tr.bad.mean():.1%} bad)", "rows": len(tr), **base}]
for ratio in [5, 3, 1]:
    T = pd.concat([bad_tr, good_tr.sample(len(bad_tr) * ratio, random_state=0)])
    rows.append({"training data": f"Undersample to {ratio}:1 good:bad ({1 / (ratio + 1):.0%} bad)", "rows": len(T), **fit_score(FEATS, Xtr=T)})
resampling = pd.DataFrame(rows).set_index("training data")
resampling.round(3)

,rows,AUC,PR-AUC,PR-AUC / random,best point where precision = recall,precision when recall >= 70%
training data,,,,,,
No resampling (7.7% bad),214087,0.754,0.198,2.999,0.242,0.130
Undersample to 5:1 good:bad (17% bad),99474,0.753,0.197,2.991,0.239,0.130
Undersample to 3:1 good:bad (25% bad),66316,0.752,0.195,2.964,0.237,0.129
Undersample to 1:1 good:bad (50% bad),33158,0.750,0.194,2.951,0.231,0.130


### Warning: what a balanced *test* set would report
If the test set is also cut to 50:50, precision and PR-AUC jump, although the model is identical (same AUC). Precision depends on how common bad loans are, and in reality 93 in 100 FY2019 loans are good. **We always report results on the real loan mix.**

In [14]:
tb = pd.concat([te[te.bad == 1], te[te.bad == 0].sample(int(te.bad.sum()), random_state=0)])
pd.DataFrame({"Real FY2019 mix (6.6% bad)": base, "Test set cut to 50:50 (misleading)": fit_score(FEATS, Xte=tb)}).T.round(3)

,AUC,PR-AUC,PR-AUC / random,best point where precision = recall,precision when recall >= 70%
Real FY2019 mix (6.6% bad),0.754,0.198,2.999,0.242,0.130
Test set cut to 50:50 (misleading),0.759,0.738,1.476,0.689,0.688


## 2. Other definitions of a bad loan
*Question:* is a different target easier to predict? (Part B lets us define the target.)
- **Early default:** charged off within 36 months of approval.
- **Large loss:** $100k or more written off.
- **Severe loss:** written-off amount of 25% or more of the loan.

In [15]:
raw = pd.read_parquet(PROC / "loans_clean.parquet")
raw = raw[raw["approvalfy"].between(2014, 2019)].reset_index(drop=True)
split = np.select([raw["approvalfy"] <= 2017, raw["approvalfy"] == 2018], ["train", "valid"], default="test")
assert (raw.loc[split == "test", "approvalfy"].values == te["approvalfy"].values).all()

co = raw["grosschargeoffamount"].fillna(0)
mco = (raw["chargeoffdate"] - raw["approvaldate"]).dt.days / 30.44
targets = {"Any charge-off (main target)": co > 0, "Early default (within 36 months)": (co > 0) & (mco <= 36),
           "Large loss ($100k or more)": co >= 1e5, "Severe loss (25%+ of the loan)": co >= 0.25 * raw["grossapproval"]}
rows = []
for name, t in targets.items():
    yt = pd.Series(t.values[split == "train"].astype(int)); ye = pd.Series(t.values[split == "test"].astype(int))
    rows.append({"target": name, "test bad rate": ye.mean(), **fit_score(FEATS, ytr=yt.values, yte=ye)})
alt_targets = pd.DataFrame(rows).set_index("target")
alt_targets.round(3)

,test bad rate,AUC,PR-AUC,PR-AUC / random,best point where precision = recall,precision when recall >= 70%
target,,,,,,
Any charge-off (main target),0.066,0.754,0.198,2.999,0.242,0.130
Early default (within 36 months),0.016,0.808,0.079,4.844,0.124,0.046
Large loss ($100k or more),0.026,0.857,0.191,7.326,0.246,0.085
Severe loss (25%+ of the loan),0.063,0.760,0.198,3.142,0.242,0.126


## 3. Outside data: state unemployment (BLS)
Source: BLS Local Area Unemployment Statistics, state monthly, seasonally adjusted, via the `jjchern/laus` GitHub data package (data to March 2019).
For each loan we take the borrower's state rate **6 months before approval** (figures are published with a delay, so this is what a lender would know), its 12-month change, and its gap from the average of all states.

In [16]:
import pyreadr
URL = "https://raw.githubusercontent.com/jjchern/laus/master/data/state_month_sa.rda"
local = EXT / "laus_state_month_sa.rda"
if not local.exists():
    EXT.mkdir(parents=True, exist_ok=True); urllib.request.urlretrieve(URL, local)
u = list(pyreadr.read_r(str(local)).values())[0]
u["date"] = pd.to_datetime(u["year"] + "-" + u["month"] + "-01")
ABB = {'Alabama':'AL','Alaska':'AK','Arizona':'AZ','Arkansas':'AR','California':'CA','Colorado':'CO','Connecticut':'CT','Delaware':'DE',
       'District of Columbia':'DC','Florida':'FL','Georgia':'GA','Hawaii':'HI','Idaho':'ID','Illinois':'IL','Indiana':'IN','Iowa':'IA',
       'Kansas':'KS','Kentucky':'KY','Louisiana':'LA','Maine':'ME','Maryland':'MD','Massachusetts':'MA','Michigan':'MI','Minnesota':'MN',
       'Mississippi':'MS','Missouri':'MO','Montana':'MT','Nebraska':'NE','Nevada':'NV','New Hampshire':'NH','New Jersey':'NJ',
       'New Mexico':'NM','New York':'NY','North Carolina':'NC','North Dakota':'ND','Ohio':'OH','Oklahoma':'OK','Oregon':'OR',
       'Pennsylvania':'PA','Rhode Island':'RI','South Carolina':'SC','South Dakota':'SD','Tennessee':'TN','Texas':'TX','Utah':'UT',
       'Vermont':'VT','Virginia':'VA','Washington':'WA','West Virginia':'WV','Wisconsin':'WI','Wyoming':'WY'}
u["st"] = u["state"].map(ABB); u = u.dropna(subset=["st"])
U = u.set_index(["st", "date"])["unem_rate"]; nat = u.groupby("date")["unem_rate"].mean()

ref = (raw["approvaldate"].dt.to_period("M") - 6).dt.to_timestamp()
ref12 = (raw["approvaldate"].dt.to_period("M") - 18).dt.to_timestamp()
ext = pd.DataFrame({"st_unemp": U.reindex(pd.MultiIndex.from_arrays([raw["borrstate"], ref])).values})
ext["st_unemp_12m_chg"] = ext["st_unemp"] - U.reindex(pd.MultiIndex.from_arrays([raw["borrstate"], ref12])).values
ext["st_unemp_vs_nat"] = ext["st_unemp"] - nat.reindex(ref).values
print(f"Loans matched to an unemployment rate: {ext['st_unemp'].notna().mean():.1%} (missing: PR, GU, VI)")
UNEMP = list(ext.columns)
for name, D in zip(["train", "valid", "test"], [tr, va, te]):
    for c in UNEMP: D[c] = ext.loc[split == name, c].values

display(tr.groupby(pd.qcut(tr["st_unemp"], 5))["bad"].agg(loans="size", bad_rate="mean").round(4))
pd.DataFrame({"Without unemployment": base, "With unemployment (3 features)": fit_score(FEATS + UNEMP)}).T.round(3)

Loans matched to an unemployment rate: 99.2% (missing: PR, GU, VI)


,loans,bad_rate
st_unemp,,
"(2.599, 4.5]",42697,0.0727
"(4.5, 5.1]",46946,0.0845
"(5.1, 5.7]",40820,0.0764
"(5.7, 6.6]",40537,0.0810
"(6.6, 10.0]",41060,0.0724


,AUC,PR-AUC,PR-AUC / random,best point where precision = recall,precision when recall >= 70%
Without unemployment,0.754,0.198,2.999,0.242,0.130
With unemployment (3 features),0.754,0.198,3.004,0.244,0.128


## 4. Outside data: an older SBA snapshot (data as of 31 Dec 2022)
The same public SBA 7(a) FOIA files, published three years earlier and going back to FY1991 (about 1.57 million loans). We use them three ways.

In [17]:
cache = PROC / "sba_snapshot_2022.parquet"
if not cache.exists():
    cols = ["BorrName", "BorrState", "BorrZip", "BankName", "GrossApproval", "ApprovalDate", "ApprovalFiscalYear", "subpgmdesc",
            "InitialInterestRate", "TermInMonths", "NaicsCode", "BusinessType", "LoanStatus", "PaidInFullDate", "ChargeOffDate",
            "GrossChargeOffAmount", "RevolverStatus", "JobsSupported"]
    files = sorted(EXT.glob("foia-7afy*-asof-221231.csv"))
    old = pd.concat([pd.read_csv(f, usecols=cols, encoding="latin1", dtype=str) for f in files], ignore_index=True)
    old.columns = [c.lower() for c in old.columns]
    for c in ["approvaldate", "paidinfulldate", "chargeoffdate"]: old[c] = pd.to_datetime(old[c], format="%m/%d/%Y", errors="coerce")
    for c in ["grossapproval", "approvalfiscalyear", "initialinterestrate", "terminmonths", "grosschargeoffamount", "jobssupported", "revolverstatus"]:
        old[c] = pd.to_numeric(old[c], errors="coerce")
    old.to_parquet(cache, index=False)
old = pd.read_parquet(cache)
print(f"{len(old):,} loans, FY{int(old.approvalfiscalyear.min())}-FY{int(old.approvalfiscalyear.max())}")

1,573,141 loans, FY1991-FY2019


### 4a. Proof that the term field is rewritten after a loan goes bad
We match FY2014-19 loans across the 2022 and 2026 snapshots (same borrower name, approval date, amount and lender) and compare the term recorded in each.

In [18]:
key = ["borrname", "approvaldate", "grossapproval", "bankname"]
o = old[old["approvalfiscalyear"].between(2014, 2019)].copy(); n = raw.copy()
n["bad"] = (n["grosschargeoffamount"].fillna(0) > 0).astype(int)
for D in (o, n): D["borrname"] = D["borrname"].astype(str).str.upper().str.strip()
m = n.drop_duplicates(key, keep=False).merge(o.drop_duplicates(key, keep=False)[key + ["terminmonths", "grosschargeoffamount"]],
                                             on=key, suffixes=("_2026", "_2022"))
m["bad_2022"] = (m["grosschargeoffamount_2022"].fillna(0) > 0).astype(int)
print(f"Matched {len(m):,} loans across both snapshots")
groups = {"Good in both": (m.bad_2022 == 0) & (m.bad == 0), "Bad in both": (m.bad_2022 == 1) & (m.bad == 1),
          "Went bad between 2022 and 2026": (m.bad_2022 == 0) & (m.bad == 1)}
pd.DataFrame({g: {"loans": int(s.sum()), "whole-year term in 2022": (m.loc[s, "terminmonths_2022"] % 12 == 0).mean(),
                  "whole-year term in 2026": (m.loc[s, "terminmonths_2026"] % 12 == 0).mean(),
                  "term changed": (m.loc[s, "terminmonths_2022"] != m.loc[s, "terminmonths_2026"]).mean()} for g, s in groups.items()}).T.round(3)

Matched 122,081 loans across both snapshots


,loans,whole-year term in 2022,whole-year term in 2026,term changed
Good in both,112247.0,0.879,0.859,0.032
Bad in both,6601.0,0.096,0.098,0.002
Went bad between 2022 and 2026,3231.0,0.593,0.129,0.609


### 4b. Borrower and lender history
- **Borrower history:** earlier SBA loans by the same borrower (cleaned name + 5-digit zip), approved before this loan. A past charge-off or pay-off counts only if it was dated before this approval, so nothing from the future is used.
- **Lender's longer track record:** each lender's charge-off rate on FY2000-09 loans, counting only charge-offs dated before 1 Oct 2013 (known before any FY2014 approval).

In [19]:
def clean_name(s):
    return (s.astype(str).str.upper().str.replace(r"[^A-Z0-9 ]", " ", regex=True)
             .str.replace(r"\b(LLC|INC|CORP|CORPORATION|CO|LTD|THE|DBA|PLLC|LLP|LP|PC|PA)\b", " ", regex=True).str.split().str.join(" "))

hist_old = old[old["approvalfiscalyear"] <= 2013]
Uall = pd.DataFrame({"name": clean_name(pd.concat([hist_old.borrname, raw.borrname])).values,
                     "zip": pd.concat([hist_old.borrzip, raw.borrzip]).astype(str).str[:5].values,
                     "appr": pd.concat([hist_old.approvaldate, raw.approvaldate]).values,
                     "co_date": pd.concat([hist_old.chargeoffdate, raw.chargeoffdate]).values,
                     "co_amt": pd.concat([hist_old.grosschargeoffamount, raw.grosschargeoffamount]).fillna(0).values,
                     "pif_date": pd.concat([hist_old.paidinfulldate, raw.paidinfulldate]).values,
                     "idx": np.r_[np.full(len(hist_old), -1), np.arange(len(raw))]})
Uall = Uall[Uall["name"].str.len() > 2]; Uall["bkey"] = Uall["name"] + "|" + Uall["zip"]
cnt = Uall["bkey"].value_counts(); V = Uall[Uall["bkey"].isin(cnt[cnt > 1].index)]
J = V[V.idx >= 0][["bkey", "appr", "idx"]].merge(V[["bkey", "appr", "co_date", "co_amt", "pif_date"]].rename(columns={"appr": "p_appr"}), on="bkey")
J = J[J.p_appr < J.appr]
J["p_co"] = (J.co_amt > 0) & (J.co_date < J.appr); J["p_pif"] = J.pif_date < J.appr
agg = J.groupby("idx").agg(prior_loans=("p_appr", "size"), prior_co=("p_co", "sum"), prior_pif=("p_pif", "sum"), last_prior=("p_appr", "max"))
H = pd.DataFrame(index=range(len(raw))).join(agg)
H["yrs_since_prior"] = (raw["approvaldate"] - H["last_prior"]).dt.days / 365
H = H.drop(columns="last_prior"); H[["prior_loans", "prior_co", "prior_pif"]] = H[["prior_loans", "prior_co", "prior_pif"]].fillna(0)
H["repeat_borrower"] = (H["prior_loans"] > 0).astype(int); H["any_prior_co"] = (H["prior_co"] > 0).astype(int)

L = old[old["approvalfiscalyear"].between(2000, 2009)].copy()
L["co_known"] = (L["grosschargeoffamount"].fillna(0) > 0) & (L["chargeoffdate"] < pd.Timestamp("2013-10-01"))
lg = L.groupby("bankname").agg(n=("co_known", "size"), co=("co_known", "sum")); pr0 = L["co_known"].mean()
H["lender_hist_rate"] = raw["bankname"].map((lg.co + 50 * pr0) / (lg.n + 50)).values
H["lender_hist_n"] = np.log10(raw["bankname"].map(lg.n).fillna(0) + 1).values

bad_all = (raw["grosschargeoffamount"].fillna(0) > 0).astype(int)
grp = np.select([H.any_prior_co == 1, H.repeat_borrower == 1], ["Earlier charge-off", "Repeat borrower, no earlier charge-off"], "First-time borrower")
display(pd.DataFrame({"bad": bad_all, "group": grp}).groupby("group")["bad"].agg(loans="size", bad_rate="mean").round(4))

BORR = ["prior_loans", "prior_co", "prior_pif", "any_prior_co", "repeat_borrower", "yrs_since_prior"]; LENDH = ["lender_hist_rate", "lender_hist_n"]
for name, D in zip(["train", "valid", "test"], [tr, va, te]):
    for c in BORR + LENDH: D[c] = H.loc[split == name, c].values
pd.DataFrame({"Baseline": base, "+ borrower history": fit_score(FEATS + BORR), "+ lender long history": fit_score(FEATS + LENDH),
              "+ both": fit_score(FEATS + BORR + LENDH)}).T.round(4)

,loans,bad_rate
group,,
Earlier charge-off,26,0.0000
First-time borrower,274775,0.0793
"Repeat borrower, no earlier charge-off",39115,0.0568


,AUC,PR-AUC,PR-AUC / random,best point where precision = recall,precision when recall >= 70%
Baseline,0.7538,0.1975,2.9994,0.2418,0.1305
+ borrower history,0.7540,0.1995,3.0285,0.2421,0.1316
+ lender long history,0.7535,0.1988,3.0186,0.2412,0.1308
+ both,0.7538,0.1988,3.0179,0.2444,0.1317


### 4c. More training years
*Question:* does training on more years help? The older files have no collateral field, so this uses a smaller shared feature set and compares training windows like-for-like.

In [20]:
o2 = old[old["approvalfiscalyear"].between(2005, 2013) & ~old["loanstatus"].isin(["CANCLD", "EXEMPT", "COMMIT"])]
shared = lambda D, fy, sub: pd.DataFrame({"fy": fy, "borrstate": D["borrstate"].values, "grossapproval": D["grossapproval"].values,
    "subprogram": sub, "rate": D["initialinterestrate"].values, "sector": D["naicscode"].astype(str).str[:2].values,
    "businesstype": D["businesstype"].values, "jobs": D["jobssupported"].values, "revolver": D["revolverstatus"].astype(float).values,
    "bad": (D["grosschargeoffamount"].fillna(0) > 0).astype(int).values})
A = pd.concat([shared(o2, o2["approvalfiscalyear"].values, o2["subpgmdesc"].values),
               shared(raw.assign(naicscode=raw["naicscode"].astype("Int64").astype(str)), raw["approvalfy"].values, raw["subprogram"].values)], ignore_index=True)
A["log_amount"] = np.log10(A["grossapproval"].clip(lower=1000)); A["rate_vs_year"] = A["rate"] - A.groupby("fy")["rate"].transform("median")
A["jobs_log"] = np.log1p(A["jobs"])
for c in ["borrstate", "subprogram", "businesstype", "sector"]: A[c] = A[c].astype("category")
SF = ["log_amount", "rate_vs_year", "jobs_log", "revolver", "borrstate", "subprogram", "businesstype", "sector"]
display(A.groupby("fy")["bad"].mean().round(3).to_frame("bad-loan rate by approval year").T)
At = A[A.fy == 2019]; rows = []
for lo in [2014, 2010, 2005]:
    Atr = A[A.fy.between(lo, 2017)]
    rows.append({"training years": f"FY{lo}-2017", "loans": len(Atr), **fit_score(SF, Xtr=Atr, Xte=At)})
pd.DataFrame(rows).set_index("training years").round(4)

fy,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019
bad-loan rate by approval year,0.252,0.331,0.379,0.316,0.154,0.097,0.072,0.066,0.065,0.072,0.077,0.079,0.081,0.081,0.066


,loans,AUC,PR-AUC,PR-AUC / random,best point where precision = recall,precision when recall >= 70%
training years,,,,,,
FY2014-2017,214087,0.7350,0.1554,2.3590,0.1994,0.1227
FY2010-2017,368419,0.7263,0.1523,2.3131,0.2045,0.1193
FY2005-2017,720090,0.7259,0.1490,2.2629,0.1961,0.1186


## Summary

| Experiment | Test AUC | Test PR-AUC | Verdict |
|---|---|---|---|
| Baseline (all 97 features) | 0.754 | 0.197 | - |
| Undersampling to 5:1, 3:1, 1:1 | 0.751-0.753 | 0.195-0.197 | No gain |
| State unemployment (BLS, 6 months before approval) | 0.753 | 0.198 | No gain |
| Borrower history (older SBA snapshot) | 0.754 | 0.200 | Tiny gain, not adopted |
| Lender long history (FY2000-09) | 0.754 | 0.199 | No gain |
| More training years (FY2005-17 vs FY2014-17, reduced features) | 0.726 vs 0.735 | 0.149 vs 0.156 | Worse |

**Findings**
- **Resampling** does not change what the model can tell apart, only its probabilities. Undersampling to 1:1 ranked slightly worse (AUC 0.751).
- **Balanced test sets mislead.** On a 50:50 test set the identical model shows PR-AUC 0.74 and about 69% precision at 70% recall. On the real FY2019 mix it is 0.20 and 13%. We always report the real mix.
- **Other targets:** large losses ($100k+) are ranked much better (AUC 0.857, PR-AUC 7.3 times random), which makes a useful second, dollar-focused view. Early default (AUC 0.81) is rarer and harder to flag precisely.
- **Unemployment** barely varies with bad-loan rates in FY2014-19 (7.2-8.5% across bands), a steady recovery, and state flags already capture regional differences.
- **Borrower history:** repeat borrowers default less (5.7% vs 7.9% for first-timers), but they are only 12% of loans, and borrowers with an earlier charge-off almost never get another SBA loan (26 cases).
- **More years hurt:** FY2005-08 loans went bad 25-38% of the time (the financial crisis), a different world from FY2014-19 (about 7%).
- **Term-field leak, proven:** of 3,231 loans that went bad between the 2022 and 2026 snapshots, 59% had a whole-year term in 2022 but only 13% in 2026; 61% had their term changed. For loans that stayed good, only 3% changed. The term is rewritten after a loan goes bad, so it cannot be used.

**Conclusion:** the public SBA data caps performance at about AUC 0.75-0.76. Reaching much higher, such as 70% precision and recall, would need borrower-level data (credit scores, cash flow, debt), which the public file does not contain.